# V2: ESM-2 Model Scaling
Compare frozen and fine-tuned ESM-2 8M, 35M, and 150M on the unchanged splits. Validation is used for selection; the test set is not evaluated here.

## 1. Repository and dependencies

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
REPO_URL = "https://github.com/yangmei25/esm2-protein-localization.git"
REPO_DIR = Path("/content/esm2-protein-localization")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

## 2. GPU, Drive, and unchanged dataset

In [ ]:
import torch, pandas as pd
from google.colab import drive, files
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU")
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/esm2-protein-localization")
DATA_PATH = DRIVE_ROOT / "data/deeploc_binary.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    uploaded = files.upload()
    if "deeploc_binary.csv" not in uploaded:
        raise ValueError("Upload data/processed/deeploc_binary.csv")
    shutil.move("deeploc_binary.csv", DATA_PATH)
data = pd.read_csv(DATA_PATH)
assert len(data) == 7890 and data["original_length"].max() <= 1022
display(data.groupby(["split", "label"]).size().unstack(fill_value=0))

## 3. Select a model
Start with `esm2_35m`. After it finishes, change only `MODEL_KEY` to `esm2_150m` and rerun from this cell.

In [ ]:
MODELS = {
    "esm2_8m": {"name": "facebook/esm2_t6_8M_UR50D", "frozen_batch": 8, "train_batch": 4, "accumulation": 4},
    "esm2_35m": {"name": "facebook/esm2_t12_35M_UR50D", "frozen_batch": 4, "train_batch": 2, "accumulation": 8},
    "esm2_150m": {"name": "facebook/esm2_t30_150M_UR50D", "frozen_batch": 2, "train_batch": 1, "accumulation": 16},
}
MODEL_KEY = "esm2_35m"
MODEL = MODELS[MODEL_KEY]
V2_ROOT = DRIVE_ROOT / "results/v2_model_scaling"
EMBEDDING_PATH = V2_ROOT / f"embeddings/{MODEL_KEY}.npz"
METADATA_PATH = V2_ROOT / f"embeddings/{MODEL_KEY}.json"
FROZEN_MODEL_DIR = V2_ROOT / f"models/frozen/{MODEL_KEY}"
FROZEN_METRICS_DIR = V2_ROOT / f"metrics/frozen/{MODEL_KEY}"
FINETUNE_DIR = V2_ROOT / f"finetune/{MODEL_KEY}_seed42"
print(MODEL_KEY, MODEL)

## 4. Extract first-token, mean, and max frozen embeddings

In [ ]:
if EMBEDDING_PATH.exists() and METADATA_PATH.exists():
    print("Reusing completed cache:", EMBEDDING_PATH)
else:
    EMBEDDING_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-u", "scripts/extract_embeddings.py", "--data", str(DATA_PATH), "--output", str(EMBEDDING_PATH), "--metadata", str(METADATA_PATH), "--model-name", MODEL["name"], "--batch-size", str(MODEL["frozen_batch"]), "--device", "cuda", "--mixed-precision", "auto"], cwd=REPO_DIR, check=True)
print(json.dumps(json.loads(METADATA_PATH.read_text()), indent=2))

## 5. Compare frozen representations on validation data

In [ ]:
SUMMARY_PATH = FROZEN_METRICS_DIR / "validation_metrics.csv"
if not SUMMARY_PATH.exists():
    subprocess.run([sys.executable, "-u", "scripts/train_embedding_classifiers.py", "--embeddings", str(EMBEDDING_PATH), "--model-dir", str(FROZEN_MODEL_DIR), "--metrics-dir", str(FROZEN_METRICS_DIR)], cwd=REPO_DIR, check=True)
display(pd.read_csv(SUMMARY_PATH))

## 6. Fine-tuning—leave disabled until both frozen runs finish
State is saved after every epoch. After a disconnect, rerun setup, set `RESUME = True`, and rerun this cell.

In [ ]:
RUN_FINETUNING = False
RESUME = False
if RUN_FINETUNING:
    command = [sys.executable, "-u", "scripts/train_finetune.py", "--data", str(DATA_PATH), "--output-dir", str(FINETUNE_DIR), "--model-name", MODEL["name"], "--epochs", "5", "--batch-size", str(MODEL["train_batch"]), "--eval-batch-size", str(max(1, MODEL["train_batch"] * 2)), "--gradient-accumulation-steps", str(MODEL["accumulation"]), "--device", "cuda", "--mixed-precision", "auto"]
    if MODEL_KEY == "esm2_150m": command.append("--gradient-checkpointing")
    if RESUME: command.append("--resume")
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print("Fine-tuning disabled until the frozen comparison is complete.")

## Next run
Record the frozen result, change `MODEL_KEY` from `esm2_35m` to `esm2_150m`, and rerun sections 3–5. Do not evaluate the test set during selection.